# 📊 Yelp Review Summarizer — Evaluation Notebook

This notebook evaluates the LLM summarization quality using:
1. **Automated metrics**: format compliance, quote faithfulness, theme coverage, hallucination detection
2. **LLM-as-Judge**: Gemini 2.5 Flash scores summaries on faithfulness, completeness, coherence, relevance, and quote accuracy
3. **Model comparison**: Compare Qwen3-0.6B vs 1.7B vs 4B

**Prerequisites**: Run `colab_runner.ipynb` first to build the FAISS index.

---

## 1️⃣ Setup

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

import os
REPO_DIR = "/content/NLP-Final--Project"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/chucey/NLP-Final--Project.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/NLP-Final--Project'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 258 (delta 73), reused 113 (delta 50), pack-reused 112 (from 1)
Receiving objects: 100% (258/258), 42.81 MiB | 19.64 MiB/s, done.
Resolving deltas: 100% (122/122), done.
Working directory: /content/NLP-Final--Project


In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    python-dotenv \
    accelerate \
    google-generativeai gensim nltk peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
# Load FAISS index from Google Drive (must have been built previously)
DRIVE_INDEX_DIR = "/content/drive/MyDrive/NLP-Project/faiss_yelp"
LOCAL_INDEX_DIR = "faiss_yelp"

if os.path.exists(DRIVE_INDEX_DIR) and os.listdir(DRIVE_INDEX_DIR):
    !cp -r {DRIVE_INDEX_DIR} {LOCAL_INDEX_DIR}
    print("✅ FAISS index loaded from Drive")
elif not os.path.exists(LOCAL_INDEX_DIR):
    print("⚠️ No FAISS index found! Run colab_runner.ipynb first to build it.")
else:
    print("✅ FAISS index already exists locally")

✅ FAISS index already exists locally


In [ ]:
# =============================================
# 🔑 Configure your Gemini API key here
# =============================================

GEMINI_API_KEY= ""
GEMINI_MODEL = "gemini-2.5-flash"
# =============================================

In [10]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

sftlist=[]
promptlist=[]
sftlist.append(""" --- Generated Summary ---
## Overall Sentiment
Positive — The reviews indicate positive experiences across multiple businesses.

## Top Praised Theme - include business name when providing evidence
- **Brio Italian Grille**: The review highlights the exceptional experience at the grill, including the variety of options, ambiance, and satisfaction with the menu.
- **Anthony's at Paxon Hollow**: The customer praises the quality of the food, the service, and mentions the possibility of returning.
- **Giordano's**: The review notes the value of the service, the comfort of the environment, and the overall satisfaction with the location.

## Top Complaints
- **Caleco's Bar & Grill**: The customer expresses disappointment with the poor taste of the food and the cost.
- **Limoncello Ristorante & Caterers**: The customer criticizes the lack of consistency in the food quality and the difficulty in ordering.
- **Moto Cucina + Enoteca**: The customer finds the service unprofessional and the food lacking in quality.

## Representative Quotes - take from Content: section of review and cite business name
1. "[exact quote from a review]" – [business name]
2. "[The atmosphere blew me away -Luigi's Pasta]" – [business name]
3. "[exact quote from a review and business being reviewed]" – [business name]

## Confidence Level
High — Based on the analysis of the reviews, I can confidently identify positive themes and address concerns effectively.
--- End Summary ---
""")
sftlist.append(""" ## Overall Sentiment
Mixed — the reviews show positive aspects, but some complaints regarding pricing, service quality, and ambiance.

## Top Praised Themes
- **Authentic Mexican Food**: Reviews praise the authenticity of dishes like grilled meats, tortillas, and seafood.
- **Good Service**: Many users appreciate friendly and attentive staff.
- **Relaxing Atmosphere**: Some comments note a welcoming environment, especially at places like *El Sabor Poblano*.

## Top Complaints
- **Overpricing**: Users express dissatisfaction with prices, particularly mentioning $250 extra for condiments.
- **Limited Parking Spaces**: A complaint highlights the difficulty in finding parking due to restricted areas.
- **Service Issues**: There are mentions of poor customer service, including lack of proper acknowledgment upon returning.

## Representative Quotes
1. "the bun makes the sonoran dog its like a snuggie for the pup a first it seems ridiculous..." – [BK Tacos]
2. "we’ve been coming here since I was a kid... the staff has watched me and now my kids grow up..." – [Casa Molina Del Norte]
3. "i would rather have a regular taco than a carnitas... the food is amazing!" – [El Sabor Poblano]

## Confidence Level
High — multiple reviews consistently highlight positive experiences, with strong recommendations and consistent stars ratings. """)
sftlist.append("""Overall Sentiment
Positive — the reviews highlight positive experiences with each business, praising the cleanliness, variety, and quality of services.

Top Praised Themes
- **Clean and inviting spaces** at Body Cycle Spinning Studio and others.
- **Diverse menu selections**, including lamb curry and steaming dishes.
- **Friendly and knowledgeable staff** at Shalimar Fine Jewelers and Ralston Massage Center.

Top Complaints
- **Inconsistent sizing issues** at Shalimar Fine Jewelers.
- **Unpleasant ambiance** at some locations.

Representative Quotes
1. "Rusells passion for fitness and cycling is so evident as is his desire for all of his clients to succeed."
2. "Wow, yummy different delicious... don't let the outside deter you."
3. "I absolutely love the experience of exploring New Orleans!"

Confidence Level
Medium — based on consistent positive feedback across multiple businesses. """)
sftlist.append("""## Overall Sentiment
Positive — Most reviews highlight positive experiences with food, service, and ambiance, though some mention issues that affected overall satisfaction.

## Top Praised Themes
- **Food Quality**: "The food come out very quickly and it is phenomenal" (Portobello Cafe), "everything tasted fresh and homemade" (Anthony's at Paxon Hollow).
- **Service**: "the waiters there funny and nice" (Portobello Cafe), "our server was good" (Caleco's Bar & Grill), "staff are as friendly as it gets" (Romano's Macaroni Grill).
- **Ambiance**: "it feels like your are entering someones home" (Portobello Cafe), "beautiful place the service adds to the charm" (Brio Italian Grille).

## Top Complaints
- **Inconsistent Food Quality**: "sometimes this food is very very good unfortunately its not consistent" (Limoncello Ristorante & Caterers), "my tomatobasil soup tasted kind of like a weak marinara" (Il Posto Italian Cafe).
- **Slow Service**: "the service was very slow" (LaScala's), "it took the waitress at least 15 minutes to get our order" (Giordano's).
- **Menu Overload/Choice Issues**: "there were too many delicious sounding items to choose from" (Anthony's at Paxon Hollow), "we didn't see a children's menu" (Anthony's Coal Fired Pizza).

## Representative Quotes
1. "The food come out very quickly and it is phenomenal definitely will be going back to this place" (Portobello Cafe)
2. "Everything tasted fresh and homemade which we both appreciated" (Anthony's at Paxon Hollow)
3. "Sometimes this food is very very good unfortunately its not consistent" (Limoncello Ristorante & Caterers)

## Confidence Level
High — Multiple reviews provide detailed feedback across various aspects of the dining experience, allowing for a comprehensive analysis. """)



promptlist=[]
promptlist.append("""  'system', 'content': 'You are a professional analyst that summarizes Yelp reviews. You MUST follow these rules strictly:\n\nRULES:\n1. ONLY use information found in the reviews provided by the user. Do NOT invent, fabricate, or assume any details.\n2. If the reviews do not contain enough information for a section, explicitly state "Not enough data."\n3. Representative quotes MUST be taken from the reviews in the |Content| section.. Do NOT paraphrase or create fake quotes.\n4. Do NOT hallucinate business names, reviewer names, menu items, or any other details not present in the reviews.\n5. If no reviews are provided, respond ONLY with: "No reviews found for the given criteria."\n\nOUTPUT FORMAT (you must follow this structure exactly):\n\n## Overall Sentiment\n[positive / mixed / negative] — one sentence explanation\n\n## Top Praised Theme - include business name when providing evidence\n- [theme 1]: brief explanation with evidence from reviews\n- [theme 2]: brief explanation with evidence from reviews\n- [theme 3]: brief explanation with evidence from reviews\n\n## Top Complaints\n- [complaint 1]: brief explanation with evidence from reviews - include business name and location when providing evidence\n- [complaint 2]: brief explanation with evidence from reviews\n- [complaint 3]: brief explanation with evidence from reviews\n\n## Representative Quotes - take from Content: section of review and cite business name\n1. "[exact quote from a review]"\n2. "[The atmosphere blew me away -Luigi\'s Pasta]"\n3. "[exact quote from a review and business being reviewed]"\n\n\n## Confidence Level\n[low / medium / high] — based on the number and consistency of reviews analyzed'}
{'role': 'user', 'content': "Please summarize the following Yelp reviews:\n\nBusiness Name: Brio Italian Grille | Content: love going here for happy hour or dinner great patio with fans to beat the stl heat alsovery accomodating at this location i like the veal milanese but with mixed greens instead of pasta theyll modify the menu to suit your taste | City: St Louis | State: MO | Review Stars: 4\n---\n\nBusiness Name: LaScala's | Content: good foodloved the gnocchi with marinara the baked eggplant appetizer was very good too the service was very slow but despite this id go back the food is just that good | City: Philadelphia | State: PA | Review Stars: 4\n---\n\nBusiness Name: Anthony's at Paxon Hollow | Content: i recently had dinner here with my wife over the weekend and could not have been more pleased our meal was excellent my wife and i were astounded by how quickly our food came out everything tasted fresh and homemade which we both appreciated the only problem was trying to pick something from the menu as there were too many delicious sounding items to choose from we will definitely be returning to try more items cant wait to see what the next chef specials will be | City: Broomall | State: PA | Review Stars: 5\n---\n\nBusiness Name: Portobello Cafe | Content: first time there and it was excellent it feels like your are entering someones home the waiters there funny and nice the food come out very quickly and it is phenomenal definitely will be going back to this place | City: Eddystone | State: PA | Review Stars: 5\n---\n\nBusiness Name: Caleco's Bar & Grill | Content: we started with the artichoke dip big mistake it was cold and the chips were stale mentioned to our waitress but nothing came of it we ordered a large pizza thin crust which after adding toppings it was a bit pricey for pizza pizza was ok not great except for not being our advocate our server was good i have been here before for cardinals game it was better then will try another venue next time | City: Saint Louis | State: MO | Review Stars: 2\n---\n\nBusiness Name: Il Posto Italian Cafe | Content: my food only merited maybe two stars but i will give an extra star because i like the menu and will try more of it and their design and concept sadly though my tomatobasil soup tasted kind of like a weak marinara and my tuna melt type sandwich had a very sharp offputting flavor it was citrusy for sure and i love lemon in a tuna melt but perhaps this was lemon zest or something just way too overpowering i think next time i will focus on the dishes with cured italian meats mortadella and prosciutto and also the baked goods those sounded pretty good especially the nutella bruschetta | City: New Orleans | State: LA | Review Stars: 3\n---\n\nBusiness Name: PizzaMan Dan's | Content: we visited once and were very disappointed in my veggie pizza and my husbands sub sandwich the tomato sauce was not tasty and they did not use enough cheese on my pizza the dough looked and tasted like it was prepared by a machine perhaps they have improved but we are not in any rush to try a second time | City: Carpinteria | State: CA | Review Stars: 1\n---\n\nBusiness Name: Ava | Content: skip this train wreck if you are looking for dinner our server was cluelesslike we seriously thought she was lost in the restaurant had to ask for setups water refills etc ordered chicken parmesan and got eggplant parmesan instead with a fish bone in it the drinks were good | City: Tampa | State: FL | Review Stars: 1\n---\n\nBusiness Name: Limoncello Ristorante & Caterers | Content: sometimes this food is very very good unfortunately its not consistent ordered something ive been getting for years and every other time it tastes incredible its like they have different people in the kitchen and you dont know who you will get so 50 of the time its excellent ugh | City: West Chester | State: PA | Review Stars: 3\n---\n\nBusiness Name: Timpano Hyde Park | Content: i had the chicken marsala recommended by our waiter the chicken was very thin tough and dry the pasta was under cooked and stuck together there was absolutely no flavor my friend ordered the lasagne and did not like it all there was only a thin layer of pasta and no flavor my other friend was still eating her dinner and they cleared her plate when she went to the restroom after she told the waiter she was not finished they did bring her a new meal in a doggie bag the positives are a great atmosphere and live music i enjoyed dancing with my husband however very expensive for awful food | City: Tampa | State: FL | Review Stars: 2\n---\n\nBusiness Name: Ava | Content: this was my first time dining at ava and i enjoyed most of it we ordered the ricotta cheese toasted bread and although the flavor was excellent i thought the bread to cheese ratio was off quite a bit the ricotta came in a tiny ramekin typically reserved for flavored butters so i expected more for an appetizer the two pizzas split among a couple of people were the sausage and the potato with rosemary they were both tasty but i think they forgot the rosemary on ours still yummy tho another entree was the grouper wisraeli couscous scrumptious dessert was a must and the zeppoles or sugared donuts were amazing the cherry tart a little less so but one person at our table loved it so maybe it was just not my cup of tea the cherries were a sort of candied cherryinteresting but sweet with a very | City: Tampa | State: FL | Review Stars: 4\n---\n\nBusiness Name: Ava | Content: it so maybe it was just not my cup of tea the cherries were a sort of candied cherryinteresting but sweet with a very dense crust again just not my personal fave i will definitely return to try other menu items that looked greatlike the pastas made inhouse dailythats next | City: Tampa | State: FL | Review Stars: 4\n---\n\nBusiness Name: Romano's Macaroni Grill | Content: great bar happy hour 47 every day wine drafts 3 5 pizza 450 apps togo dinners and lunches are very reasonable and fast staff are as friendly as it gets | City: Reno | State: NV | Review Stars: 4\n---\n\nBusiness Name: Angelina's | Content: last night we tried to get into the oldest italian restaurant in south philly thankfully they couldnt fit us in so our son said lets try angelinas so we walked around the corner and into a small italian restaurant immediatly we felt at home angelina is cooking in the front of the resaurant and her husband is servicing there were 3 of us and we started off with eggplant and roasted pepper appetizer one bite and we were hooked we ordered the veal osso bocca carbonara and lasagna everything was absolutly delish cant wait to go back | City: Philadelphia | State: PA | Review Stars: 5\n---\n\nBusiness Name: Spasso Italian Grill | Content: i went here when if first opened and have never desired to go back again i was so very disappointed at the quality and the taste of the food i would absolutely not recommend dining here | City: Media | State: PA | Review Stars: 1\n---\n\nBusiness Name: Moto Cucina + Enoteca | Content: if not for the pretentious haughty superior attitude of our waiter i would have given this place four stars or possibly more seriously that kind of attitude is exactly why i left new york we wanted to order a bottle of wine asked for his suggestion and he answers with how much do you want to spend ummm excuse me how about im happy to make some recommendations lets find a price point youre comfortable with he didnt smile once during the meal service and also found it necessary to correct us on several points of preference snob the food is good the only thing that was great was the blueberry lasagna and it is superb so was the chocolate confection for dessert id say go and i hope you dont get that waiter | City: Nashville | State: TN | Review Stars: 3\n---\n\nBusiness Name: Anthony's Coal Fired Pizza | Content: we loved the food service and decor the minute we walked in for a late lunch we were greeted with a huge smile and a warm welcome from the manager our server christina i think that was her name was delightful the beer selection was great we had the chicken rabe sandwhich and pepperoni pizza the sandwhich was very good on tasty foccacia bread the pizza was superb we love the well done taste we will definately be back one problem we may have is our 7 year old is very picky and we didnt see a chidrens menu | City: Tampa | State: FL | Review Stars: 4\n---\n\nBusiness Name: Brio Italian Grille | Content: my friends at work connoisseurs of good food in the st louis area introduced me to yet another very nice restaurant and thats brios i like everything about it starting from the complimentary bread and chips to the bruschetta pasta and salads beautiful place the service adds to the charm | City: St Louis | State: MO | Review Stars: 5\n---\n\nBusiness Name: Giordano's | Content: last time this is the second and last time my wife and i visit this place we were the only table yet it took the waitress at least 15 minutes to get our order from then our pizzas took about 45 minutes they were both small while waiting for our pizza i had to move us to the bar are a i couldnt stand listening to one of the other waitresses bitching about the place nonstop while our waitress was sweeping her bitching was amplified in that echo chamber and after all that waiting my pizza was barely warm this place has now earned a spot on my short list of places to avoid | City: Indianapolis | State: IN | Review Stars: 1\n---\n\nBusiness Name: Fellini Cafe | Content: the long and the short of it a romantic little byob in newtown square that hits the mark in several areas big portions at decent prices you wont leave hungry and now for a full review i have been to this fellinis cafe and the one in bryn mawr several times and each time ive visited there have been some memorable momentsand there have been speed bumps that keep this place from reaching its full potential on my latest visit this past friday night i made reservations for 730 my beautiful wife and i arrived right at 730 although there were two vacant tables available it took them at least 10 minutes or so to seat us and when they did they broke up a table for 6 instead of putting us at the 2 seater odd being out and away from the kids for a few hours we wanted at least a little bit of privacy | City: Newtown Square | State: PA | Review Stars: 3\n---"}
""")
promptlist.append("""'system', 'content': 'You are a professional analyst that summarizes Yelp reviews. You MUST follow these rules strictly:\n\nRULES:\n1. ONLY use information found in the reviews provided by the user. Do NOT invent, fabricate, or assume any details.\n2. If the reviews do not contain enough information for a section, explicitly state "Not enough data."\n3. Representative quotes MUST be taken from the reviews in the |Content| section.. Do NOT paraphrase or create fake quotes.\n4. Do NOT hallucinate business names, reviewer names, menu items, or any other details not present in the reviews.\n5. If no reviews are provided, respond ONLY with: "No reviews found for the given criteria."\n\nOUTPUT FORMAT (you must follow this structure exactly):\n\n## Overall Sentiment\n[positive / mixed / negative] — one sentence explanation\n\n## Top Praised Theme - include business name when providing evidence\n- [theme 1]: brief explanation with evidence from reviews\n- [theme 2]: brief explanation with evidence from reviews\n- [theme 3]: brief explanation with evidence from reviews\n\n## Top Complaints\n- [complaint 1]: brief explanation with evidence from reviews - include business name and location when providing evidence\n- [complaint 2]: brief explanation with evidence from reviews\n- [complaint 3]: brief explanation with evidence from reviews\n\n## Representative Quotes - take from Content: section of review and cite business name\n1. "[exact quote from a review]"\n2. "[The atmosphere blew me away -Luigi\'s Pasta]"\n3. "[exact quote from a review and business being reviewed]"\n\n\n## Confidence Level\n[low / medium / high] — based on the number and consistency of reviews analyzed'}
{'role': 'user', 'content': "Please summarize the following Yelp reviews:\n\nBusiness Name: BK Tacos | Content: the bun makes the sonoran dog its like a snuggie for the pup a first it seems ridiculous and almost like its going to be too much exactly like everyones favorite blanket with sleeves too much softness too much smush too indulgent wrong its warm soft chewy fragrant and it succeeds where other famed sonoran dogs fail the hot dog itself is flavorful but i would prefer that it or the bacon have a little more bite or snap to better hold their own against the dominant mustard and onions im with the masses on the carne asada caramelo excellent tortilla salty melty cheese and great carne super cheap and you can drive through | City: Tucson | State: AZ | Review Stars: 4\n---\n\nBusiness Name: El Chicanito Mexican Restaurant | Content: if you want to pay for everything a la carte this is the place for you food wasnt terrible not impressive they brought a basket of chips and some tomato sauce which i asked politely for something spicier and some pico de gallo she brought them happily to me and the salsa was much better when asked what we would like to drink i asked for a coke and she brought out a bottle which i stated i wanted the fountain drink she said oh thats only pepsi never mentioned that they only had bottle drinks for coke we ordered our food which was reasonably priced asked for sour cream and also to put cheese on the taco she let us know cheese was extra it was 250 extra for another basket of chips when i received the bill we paid more for the condiments then the actual food side of sour cream 200 pico de | City: Port Richey | State: FL | Review Stars: 1\n---\n\nBusiness Name: El Chicanito Mexican Restaurant | Content: of chips when i received the bill we paid more for the condiments then the actual food side of sour cream 200 pico de gallo 250 salsa 200 chips 250 cheese 100 and the bottled coke that we didnt want 500 just a suggestionwhen you order anything make sure to ask if there is an extra charge | City: Port Richey | State: FL | Review Stars: 1\n---\n\nBusiness Name: Jack in the Box | Content: if i could give it a zero i would i order a plain hamburger and realized they put bacon in it which i am allergic to and unable to eat after two bites when i went back to the drivethrough window to complain didnt realize the actual restaurant was openit was almost 2 after all the guy took back the burger said nothing and disappeared after 2 minutes of awkwardly making conversation with the next people in line in their car he came back and rudely told me i had to go inside to get my food which i did and still did not get an apology i refuse to go back there after that ordeal which is a shame because its nice to have a variety of places to go to after dt guess freebirds it is | City: Goleta | State: CA | Review Stars: 1\n---\n\nBusiness Name: Taqueria del Sol | Content: meh is all i can say for tds the restaurant itself is just cold not literally but figuratively there was no decor no menu boards stainless steel prep tables as dining tables no soda machines staff everywhere with no identification shirts nametags etc the food is reasonably priced and most of it was quite good but what really irked my party was the fact that they charge for extra salsa we ordered a salsa trio which included 3 samples of the various salsas in 2oz glass prep bowls and when my brother asked for more salsa for his tacos mind you we ordered one of ever taco three burritos and guacamole he was told hed have to buy the trio again really we spent over 40 on food drinks and apps too even though the food was decent i wouldnt return as the atmosphere is just blah and you cant get | City: Willow Grove | State: PA | Review Stars: 2\n---\n\nBusiness Name: Taqueria del Sol | Content: drinks and apps too even though the food was decent i wouldnt return as the atmosphere is just blah and you cant get extra salsa for your damn tacos | City: Willow Grove | State: PA | Review Stars: 2\n---\n\nBusiness Name: Cerveza Jack's | Content: this place was so chill and relaxing when i come back itll definitely be in my list if stops not to mention the artist performing was fun and eclectic i mean really a country version of let it go whats not to love | City: Nashville | State: TN | Review Stars: 5\n---\n\nBusiness Name: Chipotle Mexican Grill | Content: dont know what it is but if my tummys feeling icky i can count on chipotles stake to fix me up i just have to remember not to go during full on lunch time i usually go between 2 4 when its peaceful and calm busy or not the service is always friendly and fast | City: Tucson | State: AZ | Review Stars: 4\n---\n\nBusiness Name: Casa Molina Del Norte | Content: weve been coming here since i was a kid the staff has watched me and now my kids grow up theyre personable and so nice theyre also very prompt with service yoli and frank are the absolute best the food is amazing i absolutely love their shredded beef burro and cheese crisp i feel they have true authentic mexican food | City: Tucson | State: AZ | Review Stars: 5\n---\n\nBusiness Name: El Limon - Malvern | Content: truly i cannot say enough good things about this tiny authentic perfectly delicious eatery growing up in the panhandle of texas we had plenty of authentic mexican food in our home and im always searching for good mexican restaurants el limon is the best ive ever had in the state of pennsylvania sopes tostadas enchiladas fajitas burritos fish tacosall excellent and sooooo reasonable waitstaff is always pleasant and attentive best two features homemade tamales and free margaritas bring the entire family for an amazing mexican feast ole | City: Malvern | State: PA | Review Stars: 5\n---\n\nBusiness Name: On The Border Mexican Grill & Cantina | Content: eat here at least 2 times a week servers are always friendly and the food is amazing i never leave disappointed always leave with a full stomach | City: Mt. Laurel | State: NJ | Review Stars: 5\n---\n\nBusiness Name: Burritos Grill Mexican Fresh Cuisine | Content: cindy is the best great food and service food is authentic and so is the staff everything is made fresh and you can taste the difference from other mexican restaurants at first bite i would recommend to anyone that enjoys mexican cuisine | City: New Orleans | State: LA | Review Stars: 5\n---\n\nBusiness Name: San Jose Mexican Restaurant | Content: the service here has gone down we used to go here a lot but now we go to another one the service was slow and forgotten about | City: Lutz | State: FL | Review Stars: 2\n---\n\nBusiness Name: El Dorado | Content: they reopened up derived new management this was our 3rd time there they have daily specials which is a pretty good price 795 for a plate icetea and soup todays fridayspecial they had carne asada with rice n beans fried fish plate and 2 chicken enchiladas with rice n beans all come with tea n soup we both had the carne asada plate but they let me substitute my rice n beans for a salad steak was pretty good i think their specials are different everyday so for the most part it was good all around oh service was good n attentive | City: Tucson | State: AZ | Review Stars: 3\n---\n\nBusiness Name: Local Taco | Content: lots of improvements to this place much more comfortable environment now the chairs could be a bit more comfortable as itd be nice to sit through an entire football game otherwise this place has really improved | City: Nashville | State: TN | Review Stars: 5\n---\n\nBusiness Name: Tired Hands Fermentaria | Content: i went there for drinks and found the environment really nice with the brewery right there the beer was good and the waiter was very knowledgable and kind | City: Ardmore | State: PA | Review Stars: 4\n---\n\nBusiness Name: El Sabor Poblano | Content: although there are countless little mexican places below washington avenue i never get sick of them and always look forward to trying the new kid on the block this time it was el sabor poblano on federal which could not look less special if it tried i liked that there were 3 fish tacos on the menu i picked the fried and battered version though the tortilla chip covered fish could have been awesome i was instantly impressed when i opened one of my three and found that el sabor puts an entire filet of fish in each taco the flakey meat had no fishy taste these tacos came with the more american toppings of lettuce and tomato and cheese i bet i could have gotten cilantro and onion if i asked the sole staff member in the place seemed happy to accommodate us well we were the only ones dining in | City: Philadelphia | State: PA | Review Stars: 5\n---\n\nBusiness Name: El Sabor Poblano | Content: onion if i asked the sole staff member in the place seemed happy to accommodate us well we were the only ones dining in another taco filling option that comes with more traditional toppings is the super crispy chorizo only 8 for 3 there was no detectable gristle here something i have often come across at this little places as unassuming as it looks this particular south philly mexican place is worth stopping by | City: Philadelphia | State: PA | Review Stars: 5\n---\n\nBusiness Name: Carisma Restaurante | Content: we went to carisma restaurante again tonight it has become our go to spot the food is excellent and the service was great the owner jose is very friendly and tries to visit every table to make sure everyone is satisfied tonight we had 2 paellas salmon and a filet mignon everything was made to order cooked to perfection and came out hot the family also opened a cafe in front of the restaurant jose gave us a tour and introduced us to his daughter that runs the cafe the baked goods and coffees at the cafe are just as good as the meals at carisma good fresh food prepared and served by nice people try carisma its so much more than just the normal tacos and burritos you wont be disappointed | City: Skippack | State: PA | Review Stars: 5\n---\n\nBusiness Name: Delicia | Content: in short the food was decently good but completely overpriced the guacamole and my mojito were probably the highlight of the meal i had the peach pork medallions which were nicely cooked but lacked flavor i tried the corn cakes with barbacoa and that was pretty good the atmosphere was inviting with a hip feel if thats what youre looking for what was really annoying was the parking situation for a place with price points such as these and limited parking spaces a valet service should be provided either at a cost or complimentary dont even try and park in the back of the building its tight and dangerous even getting there and back i ended up parking three blocks away and broke a sweat walking in the dead of summer to the front entrance | City: Indianapolis | State: IN | Review Stars: 3\n---"} """)
promptlist.append("""  'system', 'content': 'You are a professional analyst that summarizes Yelp reviews. You MUST follow these rules strictly:\n\nRULES:\n1. ONLY use information found in the reviews provided by the user. Do NOT invent, fabricate, or assume any details.\n2. If the reviews do not contain enough information for a section, explicitly state "Not enough data."\n3. Representative quotes MUST be taken from the reviews in the |Content| section.. Do NOT paraphrase or create fake quotes.\n4. Do NOT hallucinate business names, reviewer names, menu items, or any other details not present in the reviews.\n5. If no reviews are provided, respond ONLY with: "No reviews found for the given criteria."\n\nOUTPUT FORMAT (you must follow this structure exactly):\n\n## Overall Sentiment\n[positive / mixed / negative] — one sentence explanation\n\n## Top Praised Theme - include business name when providing evidence\n- [theme 1]: brief explanation with evidence from reviews\n- [theme 2]: brief explanation with evidence from reviews\n- [theme 3]: brief explanation with evidence from reviews\n\n## Top Complaints\n- [complaint 1]: brief explanation with evidence from reviews - include business name and location when providing evidence\n- [complaint 2]: brief explanation with evidence from reviews\n- [complaint 3]: brief explanation with evidence from reviews\n\n## Representative Quotes - take from Content: section of review and cite business name\n1. "[exact quote from a review]"\n2. "[The atmosphere blew me away -Luigi\'s Pasta]"\n3. "[exact quote from a review and business being reviewed]"\n\n\n## Confidence Level\n[low / medium / high] — based on the number and consistency of reviews analyzed'}
{'role': 'user', 'content': "Please summarize the following Yelp reviews:\n\nBusiness Name: Moe's Southwest Grill | Content: terribly slow service as in over 30 minutes in a short line very rude employees and quite dirty very disappointed in the quality all around will not be returning to this place again | City: Nashville | State: TN | Review Stars: 1\n---\n\nBusiness Name: El Charro Cafe | Content: horrible service waited a good while and no server came or acknowledged us a couple seated after us was taken care of so after waiting another 5 min left they didnt even notice | City: Oro Valley | State: AZ | Review Stars: 1\n---\n\nBusiness Name: The C House | Content: service waswait what service we were disregarded for 15 minutes after asking for service while the quality and atmosphere may have been wonderful the service was an absolute disappointment save your money for those that appreciate your patronage | City: Tampa | State: FL | Review Stars: 1\n---\n\nBusiness Name: Claim Jumper Steakhouse & Bar | Content: service was slow forgot my order took over ten minutes to get drink after ordering which i may add it took about 10 minutes to even get the server over herenot worth the money at all | City: Tucson | State: AZ | Review Stars: 1\n---\n\nBusiness Name: Moon's Kitchen Cafe | Content: wait times were going to be much longer than expected very easy step to take the servers were clearly overworked due to managements bonehead decision to allow customers to continue to sit eventually our food did come but it wasnt worth an hour wait we wanted to like this but bad customer flow management left a bad taste in my mouth super easy fix guys dont seat people when the wait is going to be that long | City: Boise | State: ID | Review Stars: 2\n---\n\nBusiness Name: San Jose Mexican Restaurant | Content: the service here has gone down we used to go here a lot but now we go to another one the service was slow and forgotten about | City: Lutz | State: FL | Review Stars: 2\n---\n\nBusiness Name: Mavis Discount Tire | Content: worst service on the planet scheduled an appt for 100 pm service agent said it might be a brief wait because they were busy an hour and 15 minutes later my car was still not pulled into the bay for service i asked if it would get in soon and they said it would be a long wait and at at least 8 cars were ahead of me took my keys and left it was obvious they were placing all walkins ahead of me and had no concept of how to prioritize appts and had a complete disregard for customer service really didnt care that i was leaving thanks for wasting my time | City: Brookhaven | State: PA | Review Stars: 1\n---\n\nBusiness Name: Popeyes Louisiana Kitchen | Content: horrible service they didnt give us almost half of our order and when we asked for it there was no apology they werent busy and they certainly werent in any hurry we were the only ones in line and we waited almost 10 minutes to order | City: Indianapolis | State: IN | Review Stars: 1\n---\n\nBusiness Name: K's Luxury Nails | Content: too expensive for the services offered i recently moved here and based on the reviews here decided to drive 20 minutes to give this place a try the service provider that assisted me was very friendly and the wait time was a lot shorter than i normally wait but i was not given exactly what i asked for and what i did receive was not worth what i was charged i believe in paying for what i want but this was not worth the money | City: Riverview | State: FL | Review Stars: 1\n---\n\nBusiness Name: Poblanos Mexican Grill & Bar | Content: the food is delicious but the service has always been pretty slow if not bad which would be understandable if it was busy its just usually its not | City: Clearwater | State: FL | Review Stars: 3\n---\n\nBusiness Name: Sculley's Waterfront Restaurant | Content: slowest service i have ever received in my life lunch took over two hours and was just ok i will never return it was raining so we picked a restaurant quickly it was not busy not another table was seated in our room no music and our server took forever for everything menus15 minutes drinks20 minutes food30 more minutes no refills and it took forever for the check we left cash because we were afraid to give a cc because it would have taken another 30 minutes to get it back skip this place | City: Madeira Beach | State: FL | Review Stars: 1\n---\n\nBusiness Name: Salty's Island Bar & Grille | Content: slowest service ever 25 minutes after sitting down and water wasnt even delivered plenty of open tables so not sure what happened we left didnt get to try anything | City: Clearwater Beach | State: FL | Review Stars: 1\n---\n\nBusiness Name: Brunchies - Lutz | Content: the service by our server tina was greatattentive and fast service the wait was not long at all give it was a sunday around 10am 15 wait tops now to the foodmust try the huevos bruncheros best breakfast item ive had in a long time i mean i cant really describe just have to try it | City: Lutz | State: FL | Review Stars: 5\n---\n\nBusiness Name: Taco Bell | Content: the drive thru wait time is the worst in the nation minimum 15 mins with no cars in line i once waited 28 mins they are definitely 1 in poor drive thru service | City: Sparks | State: NV | Review Stars: 1\n---\n\nBusiness Name: Prep & Pastry | Content: well this place is nice place for breakfast but they are not able to provide the service probably because of volume hosts are unaware of waiting lists stuff seems like overwhelmed i think they need to improve their customer service | City: Tucson | State: AZ | Review Stars: 3\n---\n\nBusiness Name: Manhattan Bagel | Content: awful long wait times slow service out of ingredients and orders filled sloppily and incorrectly go down the street to the one on york road near the mall | City: Willow Grove | State: PA | Review Stars: 1\n---\n\nBusiness Name: McGee Auto Service & Tires | Content: excellent service quick reasonably priced went out of their way to get me in the same day because of my busy schedule will definitely be taking my car there again | City: Clearwater | State: FL | Review Stars: 5\n---\n\nBusiness Name: Q's Quik Smog | Content: super fast service went in a little after 8am when they opened and was done in about 5 mins no hassle or wait time and it was only 20 with their online coupon what more could you ask for | City: Reno | State: NV | Review Stars: 5\n---\n\nBusiness Name: Pei Wei Asian Diner | Content: i have never had such horrible service in my life i dont expect much from a chain but this is ridiculous on multiple occasions i have called in a to go order they give me a time it will be ready normally about 30mins later and i show up at that time and am forced to wait and additional 1015 minutes its chinese fast food how does it take 45 mins this place needs new management asap or this location wont be around much longer | City: Franklin | State: TN | Review Stars: 1\n---\n\nBusiness Name: Sears Auto Center | Content: the service is so bad that its an embarrassment i called three times today to find out if a certain battery is in stock i was put on hold three times for a half an hour each horrible | City: Santa Barbara | State: CA | Review Stars: 1\n---"}
""")
promptlist.append(""" 'system', 'content': 'You are a professional analyst that summarizes Yelp reviews. You MUST follow these rules strictly:\n\nRULES:\n1. ONLY use information found in the reviews provided by the user. Do NOT invent, fabricate, or assume any details.\n2. If the reviews do not contain enough information for a section, explicitly state "Not enough data."\n3. Representative quotes MUST be taken from the reviews in the |Content| section.. Do NOT paraphrase or create fake quotes.\n4. Do NOT hallucinate business names, reviewer names, menu items, or any other details not present in the reviews.\n5. If no reviews are provided, respond ONLY with: "No reviews found for the given criteria."\n\nOUTPUT FORMAT (you must follow this structure exactly):\n\n## Overall Sentiment\n[positive / mixed / negative] — one sentence explanation\n\n## Top Praised Theme - include business name when providing evidence\n- [theme 1]: brief explanation with evidence from reviews\n- [theme 2]: brief explanation with evidence from reviews\n- [theme 3]: brief explanation with evidence from reviews\n\n## Top Complaints\n- [complaint 1]: brief explanation with evidence from reviews - include business name and location when providing evidence\n- [complaint 2]: brief explanation with evidence from reviews\n- [complaint 3]: brief explanation with evidence from reviews\n\n## Representative Quotes - take from Content: section of review and cite business name\n1. "[exact quote from a review]"\n2. "[The atmosphere blew me away -Luigi\'s Pasta]"\n3. "[exact quote from a review and business being reviewed]"\n\n\n## Confidence Level\n[low / medium / high] — based on the number and consistency of reviews analyzed'}
{'role': 'user', 'content': "Please summarize the following Yelp reviews:\n\nBusiness Name: The Old Spaghetti Factory | Content: wonderful experience from start to finish the restaurant has a cozy classic feeling booths tables bar immediate seating immediate greet by a friendly helpful server food was great and such a value 3 course meal for such a reasonable price bread was so hot got a small salad great pasta all for under 13 bucks this is a super good value i recommend this for anyone that likes italian | City: Nashville | State: TN | Review Stars: 5\n---\n\nBusiness Name: Gio's | Content: a hidden gem of an italian restaurant best pasta and bread i have ever had true authentic italian atmosphere with anitalian pianist in a cozy and warm setting the service was excellent and the food superb | City: Tampa | State: FL | Review Stars: 5\n---\n\nBusiness Name: Chase Restaurant | Content: best italian in sb warm atmosphere and great staff one of the only restaurants in the area with outside seating on state st a must try | City: Santa Barbara | State: CA | Review Stars: 5\n---\n\nBusiness Name: Fellini Cafe of Media | Content: great restaurant delicious food probably one of the best italian food in the area the only one thing that i dont like its always crowded and noisy just like in italy | City: Media | State: PA | Review Stars: 4\n---\n\nBusiness Name: Tre Lune | Content: lovely italian restaurant with delicious food our service was excellent great prices in an upscale area we went dressed casual but dressy would work as well | City: Montecito | State: CA | Review Stars: 4\n---\n\nBusiness Name: Caruso's | Content: hands down my favorite restaurant the atmosphere is wonderful my husband and i always seem to make friends when we are waiting for our table the food is amazing and very reasonably priced for the amount you get if you havent eaten here yet i suggest you do especially if your a fan of italian food | City: Tucson | State: AZ | Review Stars: 5\n---\n\nBusiness Name: Luberto's Brickoven Pizza | Content: best italian restaurant there is i always get the tomato bread or wings as an app brandy cream sauce with spinach and if i have room ill get their cheesecake or tiramisu dont forget your wine its byob try their seafood specials if you get the chance i love them service and ambiance is nice too kinda dinerish but good for a date or family night | City: Dublin | State: PA | Review Stars: 5\n---\n\nBusiness Name: Limoncello Ristorante & Caterers | Content: love this place i dont get here much since my husband isnt a big fan of italian food crazy right the tables are a little cozy but i dont mind service is always friendly and efficient food is really yummy a great vibe here everyone always seems to be in a great mood and no wonder | City: West Chester | State: PA | Review Stars: 5\n---\n\nBusiness Name: Fellini Cafe of Media | Content: this is my first choice whenever we decide to go out to dinner they have a huge selection of foods that will fit any type of person who will be dining there their staff is super nice and will accommodate to whatever your needs may be the restaurant itself is beautiful and very relaxing in my opinion fellinis of media has the best italian cuisine around | City: Media | State: PA | Review Stars: 5\n---\n\nBusiness Name: Barone's Tuscan Grill | Content: as soon as you walk in you feel the authentic italian ambience from the decor to the italian servers the food is delicious and the servings are huge great place for a romantic date night | City: Moorestown | State: NJ | Review Stars: 4\n---\n\nBusiness Name: Vecchia Osteria | Content: or limoncello after a big meal i would recommend this restaurant to anyone looking for very fresh thoughtfully cooked italian food its as close as you can get to jumping on a plane and flying to italy for dinner | City: Newtown | State: PA | Review Stars: 5\n---\n\nBusiness Name: Trattoria Totaro | Content: excellent authentic italian food housemade pasta flavorful authentic dishes fantastic the desserts are worth saving room for if you can the ambiance is sosothe restaurant is small and somewhat cramped but clean functional and adequately lit it may take a while to get your food if its busy but they will keep you happy with freshly baked bread and housemade olive oil reservations are required to get into this gem i highly recommend this restaurant | City: Conshohocken | State: PA | Review Stars: 5\n---\n\nBusiness Name: Carino's Italian Ristorante | Content: by far the best italian food amazing service cozy atmosphere and family owned its a must try you wont be disappointed ask for santana absolutely amazing | City: St Pete Beach | State: FL | Review Stars: 5\n---\n\nBusiness Name: La Collina | Content: the atmosphere is excellent you feel as if youve stumbled into a stereotype of an italian restaurant live piano music mildly accented waiters extensive wine list its got a great feel to it however the food on all three occasions ive been there is good but nothing to write home or on yelp about maggianos has better and fresher food at less cost basically everything is impressive about this place except for the food which is only decent id imagine its a good place to go on a date or impress clients where the appearance of extremely fine italian dining is more important than the actual dining itself | City: Bala Cynwyd | State: PA | Review Stars: 2\n---\n\nBusiness Name: In Riva | Content: the food is well known for being the best italian around this place is just a cool setup with lots of seating space good for families with little ones | City: Philadelphia | State: PA | Review Stars: 5\n---\n\nBusiness Name: Noble Crust | Content: not a fan italian food is pretty straight forward i get trying to come up with something new but this was a swing and a miss portions were not what you would expect at an italian restaurant if you are looking for a more traditional place this isnt it seems more of a hipster hangout place was clean waitress hardly came around seemed more interested in talking to the other wait staff as the place was not busy wont be going back | City: Wesley Chapel | State: FL | Review Stars: 2\n---\n\nBusiness Name: Ralphs Italian Restaurant | Content: classic italian restaurant with authenticity as its backbone such a relaxing atmosphere along with good food to accompany you my personal favorite is the veal rollatini i recommend all to try it when i read the history of the restaurant on the placemat i was in disbelief at who had eaten there a few of the names i saw were taylor swift derek jeter and i think i even saw bill clinton on there overall id come back here again i really enjoyed my experience | City: Philadelphia | State: PA | Review Stars: 4\n---\n\nBusiness Name: Vecchia Osteria | Content: great little authentic italian restaurant we eat there regularly never had a bad meal gets really crowded on weekends so make reservations its byob which is really nice | City: Newtown | State: PA | Review Stars: 4\n---\n\nBusiness Name: Peno | Content: fantastic italian restaurant just off wydown a block east of hanley small and quaint the restaurant can probably only seat 30 or so people but its definitely worth your time i enjoyed the palermo pasta which is a papardelle with some red sauce and both sausage and ham delicious the wifey had squash ravioli and she devoured it we were stuffed so didnt have dessert but i would assume its great based off everything else | City: Clayton | State: MO | Review Stars: 4\n---\n\nBusiness Name: Romano's Macaroni Grill | Content: our fave go to italian place love the atmosphere smells wonderful and the food is excellent the hot loaf of bread is simply delicious the meal cost is good and they often have coupons best all around | City: Reno | State: NV | Review Stars: 5\n---"}
""")

from peft import LoraConfig, get_peft_model, TaskType
def prepare_sft_dataset2(collector, tok, max_length,responselist,prompt):

    try:
      ab=collector.get_labeled("good")

    except:
      print("An error found")
   # entry = {
    #        "prompt":       sft_prompt,
     #       "completion":   sft_completion,
      #      "equity":       equity,
       #     "equity_after": None,
        #    "pnl":          None,
         #   "label":        None,
          #  "timestamp":    int(time.time()),
  #}
    dictionaryList=[]
    count=0
    for g in responselist:
      thePrompt=prompt[count]
      tokenText= thePrompt+"<response> "+g
      #print(tokenText)
      print('prepare dataset debug, cmon aahah')
      tkenize=tok(tokenText,truncation=True,max_length=max_length,padding="max_length",return_tensors="pt")
      #tkenize['input_ids']
      newList=tkenize['input_ids'].cpu().numpy()[0]
      promptToken=tok(thePrompt,truncation=True,max_length=max_length,padding="max_length",return_tensors="pt")


     # newTensors=inputs2['input_ids'].cpu().numpy()[0]
   # newTensors2=inputs2['attention_mask'].cpu().numpy()[0]

      #print('mask cutoff is : ')
      #maskCutoff=len(promptToken['input_ids'].cpu().numpy()[0])
      maskCutoff=len(promptToken['input_ids'])
      #print(maskCutoff)
      count=0

      #prompt_ids = tok(entry["prompt"])["input_ids"]
      input_idsDraft=tkenize['input_ids'].cpu().numpy()[0]
      #newList2=tkenize['input_ids'].cpu().numpy()[0]
      newList2=[]
     # newList2=np.empty(len(input_idsDraft))
      #newList2=[]*len(input_idsDraft)
      print(len(newList2))
      #print(len(attention_maskDraft))

     # for a in range(0,maskCutoff):
     #   newList2[a]=-100
        #promptToken['input_ids'][a]

      #input_idsDraft=tkenize['input_ids'].cpu().numpy()[0]
      attention_maskDraft=tkenize['attention_mask'].cpu().numpy()[0]
      print(attention_maskDraft)
      count=count+1

      for a in range(0,len(attention_maskDraft)):
        if 1==attention_maskDraft[a]:
          newList2.append(-100)
        else:
          newList2.append(input_idsDraft[a])
          #newList2[a]=input_idsDraft[a]

      import numpy as np
      labelsDraft=np.array(newList2,dtype='float32')
      #np.array(my_list, dtype='float32')
      dictionaryList.append({"input_ids": input_idsDraft, "attention_mask": attention_maskDraft, "labels": labelsDraft})
      #return {"input_ids": input_idsDraft, "attention_mask": attention_maskDraft, "labels": labelsDraft

    return dictionaryList
def get_lora_model(base_model):

    print('what is up')
    from peft import LoraConfig
    #config = LoraConfig(r=8,lora_alpha=16,target_modules=["q_proj","v_proj"],lora_dropout=.005,task_type="CAUSAL_LM",bias="none")
    print('config')
    config = LoraConfig(r=8,lora_alpha=16,target_modules='all-linear',lora_dropout=.005,task_type="CAUSAL_LM",bias="none")
    print('after config')
    #peft_config = LoraConfig(task_type="SEQ_2_SEQ_LM", r=8, lora_alpha=16, lora_dropout=0.1)
    lora_model=get_peft_model(base_model, config)
    print('lora  model')
    #print(f"[LoRA r={rank}]")
    #print_trainable_params(lora_model)
    print('end)')
    return lora_model
    """
    Wrap the base model with LoRA adapters for fine-tuning.

    Returns:
        PEFT model with LoRA adapters attached.

    TODO:
    1. Create LoraConfig:
       - r=8
       - lora_alpha=16
       - target_modules=["q_proj", "v_proj"]
       - lora_dropout=0.05
       - bias="none"
       - task_type=TaskType.CAUSAL_LM
    2. Apply: peft_model = get_peft_model(base_model, config)
    3. Print trainable params: peft_model.print_trainable_parameters()
    4. Return peft_model
    """
    # TODO: Implement
    raise NotImplementedError("TODO: implement get_lora_model()")

def train_sft(peft_model, dataset, tok, epochs=3, lr=2e-4, batch_size=2):
   optimizer=torch.optim.AdamW(peft_model.parameters(), lr=lr)
  # cpu_model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.float32, device_map=torch.accelerator.current_accelerator().config)

   #peft_model=cpu_model
   #config = LoraConfig(r=8,lora_alpha=16,target_modules='all-linear',lora_dropout=.005,task_type="CAUSAL_LM",bias="none")
   #peft_model=get_peft_model(cpu_model,config)
   #peft_model.train()

   ## FIXME
   ##os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
   print(f"peft_model: {peft_model}")
   print(f"peft_model.device: {peft_model.device}")
   print(len(dataset))
   avg_loss=0
   countepoch=0
   for epoch in range(epochs):
      total_loss = 0
      #avg_loss = total_loss / max(1, len(dataset) // batch_size)
      print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")
      for i in range(0, len(dataset), batch_size):
         print('yahooo')
         batch = dataset[i:i+batch_size]
         print(f"len(dataset): {len(dataset)}")
         print(f"dataset: {dataset}")
         print(f"batch length: {len(batch)}")
         print(f"input_ids: {batch[0]['input_ids']}")
         print(f"length: input_ids: {len(batch[0]['input_ids'])},attention_mask: {len(batch[0]['attention_mask'])}, labels: {len(batch[0]['labels'])}")
         print(len(batch[0]['labels']))
         print('batch of zero')
         print(batch[0])
         input_ids = torch.tensor(batch[0]["input_ids"],dtype=torch.long).unsqueeze(0).to(peft_model.device)
         #input_ids = torch.tensor(batch[i:i+batch_size]['input_ids'],dtype=torch.long).to(peft_model.device)
         print(input_ids)
         ###input_ids = torch.tensor(batch[0]["input_ids"],dtype=torch.long)
         attention_mask = torch.tensor(batch[0]["attention_mask"],dtype=torch.long).unsqueeze(0).to(peft_model.device)
         #attention_mask = torch.tensor(batch[i:i+batch_size]["attention_mask"],dtype=torch.long).to(peft_model.device)
         print(attention_mask)
         ###attention_mask = torch.tensor(batch[0]["attention_mask"],dtype=torch.long)
         labels = torch.tensor(batch[0]["labels"],dtype=torch.long).unsqueeze(0).to(peft_model.device)
         #labels = torch.tensor(batch[i:i+batch_size]["labels"],dtype=torch.long).to(peft_model.device)
         print(labels)
         print('labels ^^^^')
         ###labels = torch.tensor(batch[0]["labels"],dtype=torch.long)
         print(f"input_ids[{len(input_ids)}]: {input_ids}, shape: {input_ids.shape}, dimention: {input_ids.ndimension()}")
         print(f"attention_mask: {attention_mask}")
         print(f"labels: {labels}")
         peft_model.print_trainable_parameters()
         #peft_model.eval()
         peft_model.train()
         #outputs = peft_model(input_ids=input_ids)
         #print(f'outputs: {outputs}')
         outputs = peft_model(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    labels=labels)
         print('how')
         loss = outputs.loss
         loss.backward()
         optimizer.step()
         optimizer.zero_grad()
         import math
         import math
         if math.isnan(loss.item()):
          total_loss+=0
         else:
          total_loss += loss.item()
         print(loss.item())
         print('average loss examination')
         countepoch=countepoch+1

      #avg_loss=total_loss / max(1, len(dataset) // batch_size)
      avg_loss=total_loss / countepoch
      print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")
  # save_dir = os.path.join(CACHE_DIR, "lora_adapter")
  # peft_model.save_pretrained(save_dir)
   peft_model.save_pretrained("lora_adapter")
  # print(f"LoRA adapter saved to {save_dir}")
   peft_model.eval()
   return
collector={"this":[1,2,3]}
dataset = prepare_sft_dataset2(collector, tok,2048,sftlist,promptlist)
print(f"Dataset ready: {len(dataset)} examples")
print('hi!')
print(dataset[0])
print(len(dataset))
        # Step 2: Create LoRA model
        #torch.cuda.empty_cache()
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
  print(f"CPU: {torch.cuda.get_device_name(0)}")
        ##os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
lora_model = get_lora_model(model)
print('does it get here?')

        # Step 3: Train
       # print(dataset[0])
print(f"device_map: {torch.accelerator.current_accelerator().type}")
        #cpu_model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.float32, device_map="cpu")

   #peft_model=cpu_model
   #     config = LoraConfig(r=8,lora_alpha=16,target_modules='all-linear',lora_dropout=.005,task_type="CAUSAL_LM",bias="none")
    #    lora_model=get_peft_model(cpu_model,config)
train_sft(lora_model, dataset, tok)
print("\nTraining complete!")




CUDA available: True
GPU: NVIDIA L4
An error found
prepare dataset debug, cmon aahah
0
[1 1 1 ... 1 1 1]
prepare dataset debug, cmon aahah
0
[1 1 1 ... 1 1 1]
prepare dataset debug, cmon aahah
0
[1 1 1 ... 1 1 1]
prepare dataset debug, cmon aahah
0
[1 1 1 ... 1 1 1]
Dataset ready: 4 examples
hi!
{'input_ids': array([  220,   364,  8948, ...,   390, 13120,   369]), 'attention_mask': array([1, 1, 1, ..., 1, 1, 1]), 'labels': array([-100., -100., -100., ..., -100., -100., -100.], dtype=float32)}
4
GPU: NVIDIA L4
what is up
config
after config
lora  model
end)
does it get here?
device_map: cuda
peft_model: PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)

## 2️⃣ Load Model & Vectorstore

In [ ]:
import rag_retrival

import sys

sys.path.append("/content/NLP-Final--Project")

#'/content/NLP-Final--Project'
from prompt import load_model, summarize_reviews
from zARCHIVE.evaluate import (
    auto_evaluate,
    llm_judge_evaluate,
    run_full_evaluation,
    evaluate_no_result_handling
)

print("Loading vectorstore...")
vs = rag_retrival.load_vectorstore()
print("✅ Vectorstore loaded!")

SyntaxError: invalid syntax (3940338163.py, line 6)

In [7]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
tok, model = load_model(MODEL_NAME)

Loading model: Qwen/Qwen3-0.6B ...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded on CUDA (dtype=torch.bfloat16)


## 3️⃣ Test: No-Result Handling

Verify that the model outputs "No reviews found" when no reviews are provided.

In [13]:
print("Testing no-result handling...")
no_result_test = evaluate_no_result_handling(summarize_reviews, tok, model)
print(f"\n{'✅ ALL PASSED' if no_result_test['all_passed'] else '❌ SOME FAILED'}")

Testing no-result handling...
  ✅ Empty string: No reviews found for the given criteria.
  ✅ Whitespace only: No reviews found for the given criteria.
  ✅ Newlines only: No reviews found for the given criteria.

✅ ALL PASSED


## 4️⃣ Define Test Cases

In [9]:
import time

# Define test cases with different filter combinations
TEST_CASES = [
    {
        "name": "Italian Restaurants (General)",
        "filters": {"categories": "Italian", "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "phrase query 2",
        "filters": {"business_name": None, "k": 20,"baseline": 'Poor service and long wait times'},
    },
    {
        "name": "phrase query 3",
        "filters": {"business_name": None, "k": 20,"baseline":"Good ethnic food in modest even shabby environment" },
    },
    {
        "name": "Mexican Restaurants",
        "filters": {"categories": "Mexican", "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "Coffee Shops",
        "filters": {"categories": "Coffee", "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "1-Star Reviews Only",
        "filters": {"review_stars": 1, "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "5-Star Reviews Only",
        "filters": {"review_stars": 5, "k": 20,"baseline": "overall customer experience food quality service atmosphere value wait time cleanliness"},
    },
    {
        "name": "No Results (Fake Business)",
        "filters": {"business_name": "ZZZZZ_NONEXISTENT_BUSINESS_12345", "k": 20},
    },
    {
        "name": "phrase query 1",
        "filters": {"business_name": None, "k": 20,"baseline":"Italian restaurant with cozy atmosphere" },
    },
    {
        "name": "phrase query 2",
        "filters": {"business_name": None, "k": 20,"baseline": 'Poor service and long wait times'},
    },
    {
        "name": "phrase query 3",
        "filters": {"business_name": None, "k": 20,"baseline":"Good ethnic food in modest even shabby environment" },
    }
]

print(f"Defined {len(TEST_CASES)} test cases")
for tc in TEST_CASES:
    print(f"  • {tc['name']}: {tc['filters']}")

Defined 11 test cases
  • Italian Restaurants (General): {'categories': 'Italian', 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • phrase query 2: {'business_name': None, 'k': 20, 'baseline': 'Poor service and long wait times'}
  • phrase query 3: {'business_name': None, 'k': 20, 'baseline': 'Good ethnic food in modest even shabby environment'}
  • Mexican Restaurants: {'categories': 'Mexican', 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • Coffee Shops: {'categories': 'Coffee', 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • 1-Star Reviews Only: {'review_stars': 1, 'k': 20, 'baseline': 'overall customer experience food quality service atmosphere value wait time cleanliness'}
  • 5-Star Reviews Only: {'review_stars': 5, 'k': 20, 'baseline': 'overall customer experience food quality se

## 5️⃣ Run Evaluation Across All Test Cases

In [ ]:
all_results = []
from gensim.models.doc2vec import Doc2Vec,\
    TaggedDocument
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
similars=[]
def similarEval(summary,docs):
  tagged_data = [TaggedDocument(words=word_tokenize(doc.lower()),
                              tags=[str(i)]) for i,
               doc in enumerate(docs)]


  themodel=Doc2Vec(vector_size=40,min_count=2,epochs=50)
  themodel.build_vocab(tagged_data)
  themodel.train(tagged_data,total_examples=themodel.corpus_count,epochs=themodel.epochs)


  document_vectors = [themodel.infer_vector(
    word_tokenize(doc.lower())) for doc in docs]


#generatedReview='hitler would have loved it here'
  generatedReview='i like the food and atmosphere is serene'

  theVector=[themodel.infer_vector(
    word_tokenize(summary.lower()))]
  a=theVector[0]
  b=document_vectors
  import numpy as np
  a2=np.sqrt(np.sum(np.square(a)))
  b2=np.sqrt(np.sum(np.square(b),axis=1))  #, axis=1

  productsN=a*b
  sums=np.sum(productsN,axis=1) #, axis=1
  productsD=a2*b2


  final=sums/productsD

  listFinal=final.tolist()
  huh=sorted(listFinal,reverse=True) #key=lambda x:x[0]
  print('array simiarity')
  #print(listFinal)
  #print(huh)
  if len(huh) < 5:
    return huh[0]
  else:
    return (huh[0]+huh[1]+huh[2]+huh[3]+huh[4])/5
adict={"business_name": None,
                "city": None,
                "state": None,
                "review_stars": None,
                "categories": None}
#baseline="overall customer experience food quality service atmosphere value wait time cleanliness"
#baseline='Poor service and long wait times'
baseline="Good ethnic food in modest even shabby environment"
docs = rag_retrival.retrieve_reviews_for_summary(
        vs,
        adict,
        baseline,
        20,False
    )

summary = summarize_reviews(docs, tok, model)
import re
import json
JUDGE_PROMPT = """\
You are an expert evaluator for text summarization systems. Your task is to evaluate a machine-generated summary of Yelp reviews.

You will be given:
1. SOURCE REVIEWS: The original reviews that were provided to the summarization model
2. GENERATED SUMMARY: The summary produced by the model

Evaluate the summary on the following 5 dimensions. For each, provide a score from 1-5 and a brief justification.

SCORING RUBRIC:

1. **Faithfulness** (Is the summary factually consistent with the source reviews?)
   - 5: Fully faithful, all claims supported by source reviews
   - 4: Mostly faithful, minor unsupported details
   - 3: Partially faithful, some unsupported claims
   - 2: Mostly unfaithful, significant fabrication
   - 1: Completely fabricated / hallucinated

2. **Completeness** (Does the summary capture the main themes?)
   - 5: Covers all major positive and negative themes
   - 4: Covers most themes, misses minor ones
   - 3: Covers some themes, misses important ones
   - 2: Very incomplete, misses most themes
   - 1: Does not address the review content at all

3. **Coherence** (Is the summary well-structured and readable?)
   - 5: Perfectly structured, clear, professional
   - 4: Well-structured with minor issues
   - 3: Somewhat organized but confusing in parts
   - 2: Poorly structured, hard to follow
   - 1: Incoherent, unreadable

4. **Relevance** (Does the summary focus on what matters?)
   - 5: Focuses on the most important aspects
   - 4: Mostly relevant, slight tangents
   - 3: Mix of relevant and irrelevant content
   - 2: Mostly irrelevant
   - 1: Completely off-topic

5. **Quote Accuracy** (Are representative quotes actually from the reviews?)
   - 5: All quotes are verbatim from the reviews
   - 4: Quotes are close paraphrases of actual reviews
   - 3: Some quotes match, others appear fabricated
   - 2: Most quotes are fabricated
   - 1: All quotes are fabricated / no quotes provided

You MUST respond ONLY with a valid JSON object in exactly this format (no markdown, no extra text):
{
    "faithfulness": {"score": <1-5>, "justification": "<brief reason>"},
    "completeness": {"score": <1-5>, "justification": "<brief reason>"},
    "coherence": {"score": <1-5>, "justification": "<brief reason>"},
    "relevance": {"score": <1-5>, "justification": "<brief reason>"},
    "quote_accuracy": {"score": <1-5>, "justification": "<brief reason>"},
    "overall_notes": "<any additional observations>"
}\
"""

def llm_judge_evaluate2(
    summary: str,
    source_reviews: str,
    api_key: str,
    model_name: str = "gemini-2.5-flash"
) -> dict:
    """Use Gemini as an LLM judge to evaluate the summary.

    Args:
        summary: The generated summary text
        source_reviews: The source reviews string
        api_key: Google AI Studio API key
        model_name: Gemini model to use (default: gemini-2.5-flash)

    Returns:
        dict with scores for each dimension and justifications
    """
    try:
        import google.generativeai as genai
    except ImportError:
        raise ImportError("Please install google-generativeai: pip install google-generativeai")

    genai.configure(api_key=api_key)

    # Truncate source reviews if too long (Gemini has token limits)
    max_source_chars = 30000
    if len(source_reviews) > max_source_chars:
        source_reviews = source_reviews[:max_source_chars] + "\n... [truncated for evaluation]"

    user_message = f"""SOURCE REVIEWS:
{source_reviews}

---

GENERATED SUMMARY:
{summary}"""

    judge_model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=JUDGE_PROMPT,
    )

    try:
        response = judge_model.generate_content(
            user_message,
            generation_config=genai.GenerationConfig(
                temperature=0.1,
                max_output_tokens=8192,
            ),
        )

        response_text = response.text.strip()
        print('response here')
        print(response_text)

        # Try to extract JSON from the response
        # Sometimes the model wraps it in ```json ... ```
        json_match = re.search(r'\{[\s\S]*\}', response_text)
        if json_match:
            result = json.loads(json_match.group())
        else:
            result = json.loads(response_text)

        # Calculate overall judge score (average of all dimensions)
        dimensions = ["faithfulness", "completeness", "coherence", "relevance", "quote_accuracy"]
        scores = [result[d]["score"] for d in dimensions if d in result]
        result["overall_judge_score"] = round(sum(scores) / len(scores), 2) if scores else 0

        return result

    except json.JSONDecodeError as e:
        return {
            "error": f"Failed to parse judge response as JSON: {e}",
            "raw_response": response_text,
            "overall_judge_score": 0,
        }
    except Exception as e:
        return {
            "error": f"Gemini API error: {e}",
            "overall_judge_score": 0,
        }
checkSummary=llm_judge_evaluate2(summary,docs,GEMINI_API_KEY)
print("what is this? 2")
print(checkSummary)
import sys
#sys.exit(0)
model=lora_model
for i, tc in enumerate(TEST_CASES):
    print(f"\n{'='*70}")
    print(f"TEST CASE {i+1}/{len(TEST_CASES)}: {tc['name']}")
    print(f"{'='*70}")

    # Retrieve reviews
    filters = tc["filters"]
    adict={"business_name": filters.get("business_name"),
                "city": filters.get("city"),
                "state": filters.get("state"),
                "review_stars": filters.get("review_stars"),
                "categories": filters.get("categories")}
    docs = rag_retrival.retrieve_reviews_for_summary(
        vs,
        adict,
        filters.get("baseline"),
        filters.get("k", 20),False
    )

    print(f"Retrieved {len(docs)} chars of review data")

    # Generate summary
    print("Generating summary...")
    start_time = time.time()
    summary = summarize_reviews(docs, tok, model)
    gen_time = time.time() - start_time
    print(f"Summary generated in {gen_time:.1f}s")
    print(f"\n--- Generated Summary ---\n{summary}\n--- End Summary ---\n")

    #quotes=fries(docs,tok,model)
    #summary=sumary+fries

    #checkSummary=llm_judge_evaluate2(summary,docs,GEMINI_API_KEY)
    #print("what is this? 2")
    #print(checkSummary)
    #theoritcally works

    #checkSummary=llm_judge_evaluate2(summary,docs,GEMINI_API_KEY)
    #print("what is this?")
    #print(checkSummary)
    #continue
    # Run evaluation
    eval_result = run_full_evaluation(
        summary, docs,
        api_key=GEMINI_API_KEY,
        gemini_model=GEMINI_MODEL
    )

    print('the similairty score')
    #print(similarEval(summary,docs))
    theScore=similarEval(summary,docs)
    print(theScore)
    print('got here after similar')
    similars.append(theScore)

    all_results.append({
        "test_case": tc["name"],
        "filters": tc["filters"],
        "summary": summary,
        "generation_time": gen_time,
        "evaluation": eval_result,
    })

    # Rate limit buffer for Gemini free tier
    time.sleep(5)

print(f"\n\n✅ All {len(TEST_CASES)} test cases completed!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


what is this? 2
{'error': 'Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 26.182704924s.', 'overall_judge_score': 0}

TEST CASE 1/11: Italian Restaurants (General)
Retrieved 10257 chars of review data
Generating summary...
Summary generated in 19.4s

--- Generated Summary ---
## Overall Sentiment  
Positive — The reviews highlight positive experiences across multiple businesses, including happy hour options, satisfying meals, and exceptional service.

## Top Praised Themes  
- **Brio Italian Grill

what is this? 2
{'error': 'Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6.122015163s.', 'overall_judge_score': 0}
Running automated evaluation...
Running LLM-as-judge evaluation (Gemini)...



📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.51/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.0/1.00
   Theme Coverage:       0.23/1.00
   Hallucination Check:  1.0/1.00
   ⚠️  Unverified quotes: ['[exact quote from a review]', '[exact quote from a review]']

❌ LLM Judge Error: Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 5.388028604s.

the similairty score
array simiarity
0.998461127281189
got here after similar

TEST CASE 2/11: phrase query 2
Retrieved 7117 chars of

what is this? 2
{'error': 'Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 24.403822189s.', 'overall_judge_score': 0}
Running automated evaluation...
Running LLM-as-judge evaluation (Gemini)...


...

what is this? 2
{'error': 'Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 53.673129877s.', 'overall_judge_score': 0}
Running automated evaluation...
Running LLM-as-judge evaluation (Gemini)...



📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.56/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.0/1.00
   Theme Coverage:       0.43/1.00
   Hallucination Check:  1.0/1.00
   ⚠️  Unverified quotes: ['[exact quote from a review]', '[exact quote from a review]']

❌ LLM Judge Error: Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 52.99765084s.

the similairty score
array simiarity
0.9990829348564148
got here after similar

TEST CASE 11/11: phrase query 3
Retrieved 8923 chars 

what is this? 2
{'error': 'Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 15.746645801s.', 'overall_judge_score': 0}
Running automated evaluation...
Running LLM-as-judge evaluation (Gemini)...



📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.3/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.0/1.00
   Theme Coverage:       0.4/1.00
   Hallucination Check:  0/1.00
   ⚠️  Unverified quotes: ['[exact quote from a review]', '[exact quote from a review]']
   ⚠️  Flags: ["Possible hallucinated business name: 'Friendly Staff'", "Possible hallucinated business name: 'Delicious Food'", "Possible hallucinated business name: 'Small Size'"]

❌ LLM Judge Error: Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gem

KeyboardInterrupt: 

## 6️⃣ Results Summary Table

In [ ]:
import pandas as pd

#case 1:
#case 2:
#CASE 3:
#case 4:
#case 5:
#case 6:
#case 7:
#case 8:
#case 9:
#every case study had an extremely high doc2vec similarity except for the fake business. There, no vocabulary was established since the model was not working with any legitmate reviews. However, the other results suggest the model accurately reflects semantics present in reviews.
rows = []
for r in all_results:
    auto = r["evaluation"].get("auto", {})
    judge = r["evaluation"].get("judge", {})

    row = {
        "Test Case": r["test_case"],
        "Gen Time (s)": round(r["generation_time"], 1),
        "Auto Score": auto.get("overall_auto_score", "N/A"),
        "Format": auto.get("format_compliance", {}).get("score", "N/A"),
        "Quotes": auto.get("quote_faithfulness", {}).get("score", "N/A"),
        "Coverage": auto.get("theme_coverage", {}).get("score", "N/A"),
        "No Halluc.": auto.get("hallucination_check", {}).get("score", "N/A"),
    }

    # Add judge scores if available
    if not judge.get("skipped") and not judge.get("error"):
        row["Judge Score"] = judge.get("overall_judge_score", "N/A")
        for dim in ["faithfulness", "completeness", "coherence", "relevance", "quote_accuracy"]:
            if dim in judge:
                row[f"J-{dim[:5].title()}"] = judge[dim]["score"]
    else:
        row["Judge Score"] = "N/A"

    rows.append(row)

results_df = pd.DataFrame(rows)
results_df['doc2vec similarity']=similars
print("\n📊 EVALUATION RESULTS SUMMARY")
print("=" * 100)
display(results_df)

# Calculate averages (excluding N/A)
numeric_cols = results_df.select_dtypes(include='number').columns
print("\n📈 AVERAGES:")
for col in numeric_cols:
    valid = results_df[col].dropna()
    if len(valid) > 0:
        print(f"   {col}: {valid.mean():.2f}")


📊 EVALUATION RESULTS SUMMARY


,Test Case,Gen Time (s),Auto Score,Format,Quotes,Coverage,No Halluc.,Judge Score,J-Faith,J-Compl,J-Coher,J-Relev,J-Quote,doc2vec similarity
0,Italian Restaurants (General),10.4,0.49,1.0,0.00,0.15,1.00,N/A,NaN,NaN,NaN,NaN,NaN,0.998480
1,Mexican Restaurants,12.2,0.44,1.0,0.33,0.55,0.00,N/A,NaN,NaN,NaN,NaN,NaN,0.998167
2,Coffee Shops,10.3,0.34,1.0,0.00,0.55,0.00,N/A,NaN,NaN,NaN,NaN,NaN,0.998670
3,1-Star Reviews Only,9.3,0.34,1.0,0.00,0.57,0.00,N/A,NaN,NaN,NaN,NaN,NaN,0.997807
4,5-Star Reviews Only,8.3,0.49,1.0,0.00,0.42,0.75,N/A,NaN,NaN,NaN,NaN,NaN,0.998903
5,No Results (Fake Business),6.2,0.45,1.0,0.00,0.00,1.00,1.4,1.0,1.0,3.0,1.0,1.0,0.249497
6,phrase query 1,11.3,0.46,1.0,0.50,0.46,0.00,N/A,NaN,NaN,NaN,NaN,NaN,0.998705
7,phrase query 2,9.3,0.67,1.0,0.25,0.57,1.00,N/A,NaN,NaN,NaN,NaN,NaN,0.999029
8,phrase query 3,10.1,0.30,1.0,0.00,0.40,0.00,N/A,NaN,NaN,NaN,NaN,NaN,0.998916



📈 AVERAGES:
   Gen Time (s): 9.71
   Auto Score: 0.44
   Format: 1.00
   Quotes: 0.12
   Coverage: 0.41
   No Halluc.: 0.42
   J-Faith: 1.00
   J-Compl: 1.00
   J-Coher: 3.00
   J-Relev: 1.00
   J-Quote: 1.00
   doc2vec similarity: 0.92


## 7️⃣ Model Comparison (0.6B vs 1.7B vs 4B)

Compare summarization quality across different Qwen3 model sizes.

> ⚠️ This section loads multiple models sequentially. Each model download may take a few minutes the first time.

In [ ]:
import gc

MODELS_TO_COMPARE = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-1.7B",
    "Qwen/Qwen3-4B",
]

# Use one consistent test case for fair comparison
COMPARISON_FILTERS = {"categories": "Italian", "k": 80}

addDict={"business_name": None,
                "city": None,
                "state": None,
                "review_stars": None,
                "categories": "Italian"}
print("Retrieving reviews for comparison test...")
baseline="good customer service and large wine selection"
comparison_docs = rag_retrival.retrieve_reviews_for_summary(
    vs, addDict,baseline,20,False
)
print(f"Retrieved {len(comparison_docs)} chars of review data")

model_results = []

for model_name in MODELS_TO_COMPARE:
    print(f"\n{'='*60}")
    print(f"🤖 Loading {model_name}...")
    print(f"{'='*60}")

    try:
        tok_cmp, model_cmp = load_model(model_name)

        print("Generating summary...")
        start_time = time.time()
        summary = summarize_reviews(comparison_docs, tok_cmp, model_cmp)
        gen_time = time.time() - start_time
        print(f"Generated in {gen_time:.1f}s")
        print(f"\n--- Summary ---\n{summary}\n--- End ---\n")

        # Evaluate
        eval_result = run_full_evaluation(
            summary, comparison_docs,
            api_key=GEMINI_API_KEY,
            gemini_model=GEMINI_MODEL
        )

        model_results.append({
            "model": model_name,
            "summary": summary,
            "gen_time": gen_time,
            "evaluation": eval_result,
        })
        theScore=similarEval(summary,docs)
        print(theScore)
        print('got here after similar')
        # Free GPU memory before loading next model
        del model_cmp, tok_cmp
        gc.collect()
        torch.cuda.empty_cache()

        time.sleep(5)  # Rate limit buffer

    except Exception as e:
        print(f"❌ Failed to load {model_name}: {e}")
        model_results.append({
            "model": model_name,
            "error": str(e),
        })

print(f"\n✅ Model comparison complete!")

Retrieving reviews for comparison test...
Retrieved 10257 chars of review data

🤖 Loading Qwen/Qwen3-0.6B...
Loading model: Qwen/Qwen3-0.6B ...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on CUDA (dtype=torch.bfloat16)
Generating summary...
Generated in 13.4s

--- Summary ---
## Overall Sentiment  
Positive — The reviews highlight positive experiences across multiple businesses, including exceptional food, welcoming ambiance, and customer satisfaction.

## Top Praised Themes  
1. **Exceptional Food**: All reviewed restaurants feature top-rated meals such as venison Milanese, grilled cheese sandwiches, and artisanal desserts.  
2. **Widespread Community Atmosphere**: Many reviews mention lively settings and a strong sense of community, making them enjoyable for diners.  
3. **Friendly Service**: Positive feedback includes praise for attentive staff and efficient operations, contributing to overall satisfaction.

## Top Complaints  
1. **Slow Service**: Some reviews note delays in processing orders and response times, particularly affecting customers who need immediate assistance.  
2. **Overwhelming Menu Options**: A few comments express frustration with the


📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.3/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.0/1.00
   Theme Coverage:       0.38/1.00
   Hallucination Check:  0/1.00
   ⚠️  Unverified quotes: ['[Brio Italian Grille] my friends at work connoisseurs of good food in the St Louis area introduced me to yet another very nice restaurant...', "[Anthony's at Paxon Hollow] our meal was excellent...we both appreciated the only problem was trying to pick something from the menu..."]
   ⚠️  Flags: ["Possible hallucinated business name: 'Exceptional Food'", "Possible hallucinated business name: 'Widespread Community Atmosphere'", "Possible hallucinated business name: 'Friendly Service'"]

❌ LLM Judge Error: Gemini API error: 403 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Your API key was reported as leaked. Please use another API key.


🤖 Loading Qwen/Qwen3-1.7B...
Loading model: Qwen

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded on CUDA (dtype=torch.bfloat16)
Generating summary...
Generated in 23.0s

--- Summary ---
## Overall Sentiment
Positive — Most reviews highlight positive experiences with food, service, and ambiance, though some mention issues that affected overall satisfaction.

## Top Praised Themes
- **Food Quality**: "The food come out very quickly and it is phenomenal" (Portobello Cafe), "everything tasted fresh and homemade" (Anthony's at Paxon Hollow).
- **Service**: "the waiters there funny and nice" (Portobello Cafe), "our server was good" (Caleco's Bar & Grill), "staff are as friendly as it gets" (Romano's Macaroni Grill).
- **Ambiance**: "it feels like your are entering someones home" (Portobello Cafe), "beautiful place the service adds to the charm" (Brio Italian Grille).

## Top Complaints
- **Inconsistent Food Quality**: "sometimes this food is very very good unfortunately its not consistent" (Limoncello Ristorante & Caterers), "my tomatobasil soup tasted kind of like a weak m


📊 EVALUATION REPORT

🤖 Automated Metrics (Overall: 0.53/1.00)
   Format Compliance:    1.0/1.00
   Quote Faithfulness:   0.71/1.00
   Theme Coverage:       0.46/1.00
   Hallucination Check:  0/1.00
   ⚠️  Unverified quotes: ["we didn't see a children's menu", 's at Paxon Hollow).\n- **Service**: "the waiters there funny and nice" (Portobello Cafe), "our server was good" (Caleco']
   ⚠️  Flags: ["Possible hallucinated business name: 'Food Quality'", "Possible hallucinated business name: 'Inconsistent Food Quality'", "Possible hallucinated business name: 'Slow Service'"]

❌ LLM Judge Error: Gemini API error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exc

In [ ]:
# Build comparison table
comp_rows = []
for r in model_results:
    if "error" in r:
        comp_rows.append({"Model": r["model"], "Error": r["error"]})
        continue

    auto = r["evaluation"].get("auto", {})
    judge = r["evaluation"].get("judge", {})

    row = {
        "Model": r["model"].split("/")[-1],
        "Gen Time (s)": round(r["gen_time"], 1),
        "Auto Score": auto.get("overall_auto_score", "N/A"),
        "Format": auto.get("format_compliance", {}).get("score", "N/A"),
        "Quotes": auto.get("quote_faithfulness", {}).get("score", "N/A"),
        "Coverage": auto.get("theme_coverage", {}).get("score", "N/A"),
        "No Halluc.": auto.get("hallucination_check", {}).get("score", "N/A"),
    }

    if not judge.get("skipped") and not judge.get("error"):
        row["Judge Score"] = judge.get("overall_judge_score", "N/A")
        for dim in ["faithfulness", "completeness", "coherence", "relevance", "quote_accuracy"]:
            if dim in judge:
                row[f"J-{dim[:5].title()}"] = judge[dim]["score"]

    comp_rows.append(row)
#the similairties are high for every case except the fake business. The doc2vec model could not build a sufficiently detailed vocabulary in the absence of a proper summary. However, the score was extremely high in all other instances. It seems the model captured both semantic meaning and the word choice of retrieved reviews.
comp_df = pd.DataFrame(comp_rows)
print("\n📊 MODEL COMPARISON RESULTS")
print("=" * 100)
print(f"Test case: {COMPARISON_FILTERS}")
display(comp_df)


📊 MODEL COMPARISON RESULTS
Test case: {'categories': 'Italian', 'k': 80}


,Model,Gen Time (s),Auto Score,Format,Quotes,Coverage,No Halluc.
0,Qwen3-0.6B,13.4,0.30,1.0,0.00,0.38,0
1,Qwen3-1.7B,15.7,0.44,1.0,0.29,0.62,0
2,Qwen3-4B,23.0,0.53,1.0,0.71,0.46,0


## 8️⃣ Save Results

In [ ]:
import json

# Save results to Google Drive
save_dir = "/content/drive/MyDrive/NLP-Project/eval_results"
os.makedirs(save_dir, exist_ok=True)

# Save test case results
results_df.to_csv(f"{save_dir}/test_case_results.csv", index=False)

# Save model comparison results
comp_df.to_csv(f"{save_dir}/model_comparison_results.csv", index=False)

# Save detailed results as JSON
serializable_results = []
for r in all_results:
    sr = {k: v for k, v in r.items() if k != 'evaluation'}
    sr['evaluation'] = r['evaluation']
    serializable_results.append(sr)

with open(f"{save_dir}/detailed_results.json", "w") as f:
    json.dump(serializable_results, f, indent=2, default=str)

print(f"✅ Results saved to {save_dir}/")
print(f"   • test_case_results.csv")
print(f"   • model_comparison_results.csv")
print(f"   • detailed_results.json")